# News-Based Next-Month Demand Prediction

This notebook trains an XGBoost model using `clean_english.csv` or `final_news_dataset_cleaned_english.csv`.

**Important:** the CSV has no real demand quantity/import-volume target. Therefore this notebook creates a transparent **news-based demand probability proxy** from shortage, production, price, sentiment, export-opportunity, and confidence signals. XGBoost estimates the monthly signal and a recent-trend projection forecasts the next month. The output is suitable for recommendation/ranking, not tonnes or exact purchase quantity.

In [ ]:
# Run this once in your terminal if packages are missing:
# pip install pandas numpy scikit-learn xgboost joblib groq jupyter

from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

SEARCH_DIRS = [Path.cwd(), Path.cwd() / "Demand_prediction", Path.cwd().parent / "Demand_prediction"]
DATA_NAMES = ["clean_english.csv", "final_news_dataset_cleaned_english.csv"]

DATA_PATH = None
for folder in SEARCH_DIRS:
    for name in DATA_NAMES:
        candidate = folder / name
        if candidate.exists():
            DATA_PATH = candidate.resolve()
            break
    if DATA_PATH is not None:
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Put clean_english.csv or final_news_dataset_cleaned_english.csv "
        "in the same Demand_prediction folder as this notebook."
    )

OUTPUT_DIR = DATA_PATH.parent
MODEL_PATH = OUTPUT_DIR / "demand_model_bundle.joblib"
MONTHLY_PATH = OUTPUT_DIR / "monthly_demand_features.csv"
PREDICTIONS_PATH = OUTPUT_DIR / "next_month_demand_predictions.csv"
TOP3_PATH = OUTPUT_DIR / "top3_demand_recommendations.csv"

print("Using dataset:", DATA_PATH)
print("Outputs will be saved in:", OUTPUT_DIR)

In [ ]:
# 1. Load and validate the news dataset
news = pd.read_csv(DATA_PATH)

required_columns = {
    "commodity", "country", "date", "sentiment", "sentiment_score",
    "shortage_flag", "production_drop", "production_rise",
    "price_increase", "price_decrease", "export_opportunity_score",
    "confidence",
}
missing = required_columns.difference(news.columns)
if missing:
    raise ValueError(f"Dataset is missing columns: {sorted(missing)}")

news["date"] = pd.to_datetime(news["date"], errors="coerce")
news = news.dropna(subset=["date", "country", "commodity"]).copy()
news["country"] = news["country"].astype(str).str.strip()
news["commodity"] = news["commodity"].astype(str).str.strip()
news["sentiment"] = news["sentiment"].fillna("neutral").astype(str).str.lower().str.strip()

numeric_columns = [
    "sentiment_score", "shortage_flag", "production_drop", "production_rise",
    "price_increase", "price_decrease", "export_opportunity_score", "confidence",
]
for column in numeric_columns:
    news[column] = pd.to_numeric(news[column], errors="coerce").fillna(0.0)

news["sentiment_score"] = news["sentiment_score"].clip(-1.0, 1.0)
news["export_opportunity_score"] = news["export_opportunity_score"].clip(0.0, 100.0)
news["confidence"] = news["confidence"].clip(0.0, 1.0)
for column in ["shortage_flag", "production_drop", "production_rise", "price_increase", "price_decrease"]:
    news[column] = news[column].clip(0.0, 1.0)

news["month"] = news["date"].dt.to_period("M")

print("Rows:", len(news))
print("Date range:", news["date"].min().date(), "to", news["date"].max().date())
print("Countries:", news["country"].nunique())
print("Commodities:", news["commodity"].nunique())
display(news.head())

In [ ]:
# 2. Build a transparent row-level demand probability proxy
negative_pressure = (-news["sentiment_score"]).clip(lower=0.0)
positive_sentiment = news["sentiment_score"].clip(lower=0.0)

news["article_demand_signal"] = (
    0.25
    + 0.25 * news["shortage_flag"]
    + 0.20 * news["production_drop"]
    - 0.12 * news["production_rise"]
    + 0.14 * news["price_increase"]
    - 0.10 * news["price_decrease"]
    + 0.20 * (news["export_opportunity_score"] / 100.0)
    + 0.08 * news["confidence"]
    + 0.08 * negative_pressure
    - 0.03 * positive_sentiment
).clip(0.01, 0.99)

# Higher-confidence articles receive more weight.
news["signal_weight"] = 0.25 + news["confidence"]
news["weighted_signal"] = news["article_demand_signal"] * news["signal_weight"]
news["positive_value"] = (news["sentiment"] == "positive").astype(float)
news["negative_value"] = (news["sentiment"] == "negative").astype(float)
news["neutral_value"] = (~news["sentiment"].isin(["positive", "negative"])).astype(float)

display(news[[
    "country", "commodity", "date", "sentiment_score", "shortage_flag",
    "production_drop", "export_opportunity_score", "confidence",
    "article_demand_signal"
]].head(10))

In [ ]:
# 3. Aggregate article signals by country, commodity, and month
monthly = (
    news.groupby(["country", "commodity", "month"], as_index=False)
    .agg(
        article_count=("date", "size"),
        average_sentiment=("sentiment_score", "mean"),
        positive_share=("positive_value", "mean"),
        negative_share=("negative_value", "mean"),
        neutral_share=("neutral_value", "mean"),
        shortage_share=("shortage_flag", "mean"),
        production_drop_share=("production_drop", "mean"),
        production_rise_share=("production_rise", "mean"),
        price_increase_share=("price_increase", "mean"),
        price_decrease_share=("price_decrease", "mean"),
        average_export_opportunity=("export_opportunity_score", "mean"),
        average_confidence=("confidence", "mean"),
        weighted_signal_sum=("weighted_signal", "sum"),
        total_weight=("signal_weight", "sum"),
    )
)
monthly["current_demand_probability"] = (
    monthly["weighted_signal_sum"] / monthly["total_weight"].replace(0, np.nan)
).fillna(0.5).clip(0.01, 0.99)

monthly = monthly.sort_values(["country", "commodity", "month"]).reset_index(drop=True)
monthly["previous_month"] = monthly.groupby(["country", "commodity"])["month"].shift(1)
monthly["previous_probability"] = monthly.groupby(["country", "commodity"])["current_demand_probability"].shift(1)
monthly["previous_article_count"] = monthly.groupby(["country", "commodity"])["article_count"].shift(1)

month_ordinal = monthly["month"].map(lambda value: value.ordinal).astype(float)
previous_ordinal = monthly["previous_month"].map(
    lambda value: value.ordinal if pd.notna(value) else np.nan
)
consecutive_previous = monthly["previous_month"].notna() & (
    month_ordinal - previous_ordinal == 1
)
monthly["demand_lag1"] = np.where(
    consecutive_previous,
    monthly["previous_probability"],
    monthly["current_demand_probability"],
)
monthly["demand_roll2"] = (
    monthly["current_demand_probability"] + monthly["demand_lag1"]
) / 2.0
monthly["article_growth"] = np.where(
    consecutive_previous,
    (monthly["article_count"] - monthly["previous_article_count"])
    / monthly["previous_article_count"].clip(lower=1.0),
    0.0,
)
monthly["article_growth"] = monthly["article_growth"].replace([np.inf, -np.inf], 0.0).clip(-5.0, 5.0)
monthly["year"] = monthly["month"].dt.year.astype(int)
monthly["month_number"] = monthly["month"].dt.month.astype(int)

monthly.to_csv(MONTHLY_PATH, index=False)
print("Saved:", MONTHLY_PATH)
display(monthly.head(10))

In [ ]:
# 4. Prepare the XGBoost demand-signal model
# The model learns the monthly news-derived demand probability.
# A separate recent-trend step will project this signal to the next month.

categorical_features = ["country", "commodity"]
numeric_features = [
    "article_count", "average_sentiment", "positive_share", "negative_share",
    "neutral_share", "shortage_share", "production_drop_share",
    "production_rise_share", "price_increase_share", "price_decrease_share",
    "average_export_opportunity", "average_confidence",
    "demand_lag1", "article_growth", "year", "month_number",
]
model_features = categorical_features + numeric_features

X = monthly[model_features].copy()
y = monthly["current_demand_probability"].astype(float)

latest_month = monthly["month"].max()
test_mask = monthly["month"] == latest_month
train_mask = ~test_mask

if train_mask.sum() < 20 or test_mask.sum() < 5:
    split_index = max(1, int(len(monthly) * 0.8))
    ordered = monthly.sort_values("month").index
    train_indices = ordered[:split_index]
    test_indices = ordered[split_index:]
    train_mask = monthly.index.isin(train_indices)
    test_mask = monthly.index.isin(test_indices)

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Held-out month:", latest_month)

In [ ]:
# 5. Train the XGBoost model
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numeric_features),
    ]
)

regressor = XGBRegressor(
    n_estimators=350,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=2,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="reg:squarederror",
    reg_alpha=0.05,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", regressor),
])
model.fit(X_train, y_train)

pred = np.clip(model.predict(X_test), 0.01, 0.99)
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred) if len(y_test) >= 2 else float("nan")

print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R2:   {r2:.4f}")
print("Note: metrics measure estimation of the NEWS-DERIVED monthly signal, not real demand quantity.")

In [ ]:
# 6. Refit on all available month-to-next-month examples and save the model bundle
model.fit(X, y)

bundle = {
    "model": model,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "model_features": model_features,
    "data_file": DATA_PATH.name,
    "method": "XGBoost monthly demand signal + recent trend projection",
    "metrics": {
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": float(r2),
        "test_rows": int(len(y_test)),
    },
}
joblib.dump(bundle, MODEL_PATH)
print("Saved model:", MODEL_PATH)

In [ ]:
# 7. Forecast the next month for every latest country + commodity pair
forecast_rows = []

for (country, commodity), group in monthly.groupby(["country", "commodity"]):
    group = group.sort_values("month").copy()
    latest = group.iloc[-1].copy()

    current_model_probability = float(np.clip(
        model.predict(pd.DataFrame([latest[model_features]]))[0], 0.01, 0.99
    ))

    history_values = group["current_demand_probability"].tail(4).to_numpy(dtype=float)
    history_values[-1] = current_model_probability

    if len(history_values) >= 2:
        x = np.arange(len(history_values), dtype=float)
        trend_slope = float(np.polyfit(x, history_values, 1)[0])
    else:
        trend_slope = 0.0
    trend_slope = float(np.clip(trend_slope, -0.10, 0.10))

    predicted = float(np.clip(
        current_model_probability + 0.65 * trend_slope,
        0.01,
        0.99,
    ))

    forecast_rows.append({
        "forecast_month": str(latest["month"] + 1),
        "country": country,
        "commodity": commodity,
        "predicted_demand_probability": predicted,
        "predicted_demand_percentage": round(predicted * 100.0, 2),
        "current_demand_probability": current_model_probability,
        "probability_change_points": round((predicted - current_model_probability) * 100.0, 2),
        "trend_slope_per_month": trend_slope,
        "article_count": int(latest["article_count"]),
        "average_confidence": float(latest["average_confidence"]),
        "average_export_opportunity": float(latest["average_export_opportunity"]),
        "shortage_share": float(latest["shortage_share"]),
        "production_drop_share": float(latest["production_drop_share"]),
    })

all_predictions = pd.DataFrame(forecast_rows)


def direction(probability):
    if probability >= 0.70:
        return "Strong Increase"
    if probability >= 0.58:
        return "Increase"
    if probability >= 0.45:
        return "Stable"
    if probability >= 0.32:
        return "Decrease"
    return "Strong Decrease"

all_predictions["predicted_direction"] = all_predictions["predicted_demand_probability"].apply(direction)
all_predictions["reliability_score"] = (
    0.55 * all_predictions["average_confidence"]
    + 0.45 * np.minimum(np.log1p(all_predictions["article_count"]) / np.log(20), 1.0)
).clip(0.0, 1.0)

all_predictions = all_predictions.sort_values(
    ["predicted_demand_probability", "reliability_score"], ascending=False
)
all_predictions.to_csv(PREDICTIONS_PATH, index=False)

all_predictions["country_rank"] = all_predictions.groupby("country")[
    "predicted_demand_probability"
].rank(method="first", ascending=False).astype(int)
top3 = all_predictions[all_predictions["country_rank"] <= 3].sort_values(
    ["country", "country_rank"]
)
top3.to_csv(TOP3_PATH, index=False)

print("Saved predictions:", PREDICTIONS_PATH)
print("Saved top-3 recommendations:", TOP3_PATH)
display(top3.head(30))

## Files created by this notebook

- `demand_model_bundle.joblib` — trained model used by `demand_agent_tools.py`
- `monthly_demand_features.csv` — monthly engineered features
- `next_month_demand_predictions.csv` — predictions for all country/commodity pairs
- `top3_demand_recommendations.csv` — top three commodities for each country

After these files are created, run `python demand_agent.py` from the same folder.